In [ ]:
import os
import numpy as np
import pickle
from ultralytics import YOLO
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import cv2

# 1. Initialize the ultra-fast Nano Pose Engine
pose_model = YOLO("yolo26n-pose.pt")

DATA_DIR = "../dataset"
X = []
y = []

print("📂 Extracting skeletal coordinates from your photos...")

# 2. Loop through every letter folder
for letter_folder in os.listdir(DATA_DIR):
    folder_path = os.path.join(DATA_DIR, letter_folder)
    if not os.path.isdir(folder_path):
        continue

    for img_name in os.listdir(folder_path):
        img_path = os.path.join(folder_path, img_name)

        # Read the image safely
        img = cv2.imread(img_path)
        if img is None:
            continue

        # Run inference using the pose engine
        results = pose_model(img, verbose=False)

        for r in results:
            # Check if a hand skeleton was successfully detected
            if r.keypoints is not None and len(r.keypoints.xyn) > 0:
                # Get normalized (x, y) coordinates for the 21 joints
                # xyn shape is (num_hands, 21, 2)
                landmarks = r.keypoints.xyn[0].cpu().numpy()

                # Flatten the 21 x 2 matrix into a simple list of 42 numbers
                flattened_features = landmarks.flatten()

                # Double-check that we captured all 21 joints completely
                if len(flattened_features) == 42:
                    X.append(flattened_features)
                    y.append(letter_folder)

# 3. Convert collected lists into structured numpy arrays
X = np.array(X)
y = np.array(y)

print(f"✅ Successfully processed {len(X)} images!")

# 4. Split data into training and validation sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 5. Train a lightning-fast Random Forest Classifier
print("🤖 Training the gesture classifier...")
classifier = RandomForestClassifier(n_estimators=100, random_state=42)
classifier.fit(X_train, y_train)

# 6. Evaluate accuracy metrics
predictions = classifier.predict(X_test)
score = accuracy_score(y_test, predictions)
print(f"🎯 Validation Set Accuracy Score: {score:.2%}")

# 7. Save the trained weight matrix as a tiny file (under 2MB!)
with open("gesture_classifier.pkl", "wb") as f:
    pickle.dump(classifier, f)
print("💾 Saved 'gesture_classifier.pkl' successfully!")